In [1]:
import os
import torch
from torch.utils.data import DataLoader
from pathlib import Path
import sys

project_root = Path(os.getcwd()).parent
print(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.utils.seed import set_seed
set_seed(42)

from src.data.preprocessing.pipeline import Pipeline
from src.data.datasets.universal_dataset import CVADataset
from src.models.network import DiffusionSSSD
from src.models.gaussian_noise import GaussianDiffusion
from src.train.trainer import setup_optimizer, DiffusionTrainer


/mnt/c/Users/edtop/ITMO/THESIS_CV_DIFF


CUDA extension for structured kernels (Cauchy and Vandermonde multiplication) not found. Install by going to extensions/kernels/ and running `python setup.py install`, for improved speed and memory efficiency. Note that the kernel changed for state-spaces 4.0 and must be recompiled.
Falling back on slow Cauchy and Vandermonde kernel. Install at least one of pykeops or the CUDA extension for better speed and memory efficiency.
/mnt/c/Users/edtop/ITMO/THESIS_CV_DIFF/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Global hyperpar
EPOCHS = 100
BATCH_SIZE = 64
LR = 0.00095
WEIGHT_DECAY = 0.01
TIMESTEPS = 1000 

TEST_INHIBITOR = "2-mercaptobenzimidazole" 

NUM_CYCLE = [1, 2, 3, 4]
save_dir = project_root / "experiments" / "run_01"
SAVE_DIR = str(save_dir)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[*] Device: {DEVICE}")

[*] Device: cuda


In [3]:
pipe = Pipeline(
    num_cycle=NUM_CYCLE, 
    test_inhibitor=TEST_INHIBITOR, 
    norm_feat=True, 
    use_wavelet=False
)

train_dataset = CVADataset(
    vol=pipe.train_voltage,
    cur=pipe.train_current,
    desc_df=pipe.train_analyzed_data
)

val_dataset = CVADataset(
    vol=pipe.test_voltage,
    cur=pipe.test_current,
    desc_df=pipe.test_analyzed_data
)

In [4]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
print(f"Size Train: {len(train_dataset)} samples")
print(f"Size Val: {len(val_dataset)} samples")

Size Train: 2684 samples
Size Val: 776 samples


In [5]:
num_desc_features = train_dataset[0]["features"].shape[0]

net = DiffusionSSSD(
        in_channels=1, 
        desc_features=num_desc_features, 
        base_channels=64
    )
    
diffusion = GaussianDiffusion(model=net, timesteps=TIMESTEPS)

optimizer, scheduler = setup_optimizer(
    model=net, 
    lr=LR, 
    weight_decay=WEIGHT_DECAY, 
    epochs=EPOCHS
)

Disabling PyTorch because PyTorch >= 2.4 is required but found 2.1.2+cu121
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [ ]:
trainer = DiffusionTrainer(
        diffusion_model=diffusion,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        scheduler=scheduler,
        device=DEVICE,
        save_dir=SAVE_DIR, 
        vol_scaler=pipe.vol_scaler,
        cur_scaler=pipe.cur_scaler
    )

print("\n" + "="*40)
print("Start")
print("="*40)
trainer.fit(epochs=EPOCHS)


Start
Teaching on cuda...


Epoch 1 [Val]: 100%|██████████| 13/13 [00:03<00:00,  4.09it/s, val_loss=0.0472]


Epoch 1 | Train Loss: 0.0706 | Val Loss: 0.0475 | LR: 0.001200
Saved best model (Val Loss: 0.0475)


Sampling: 100%|██████████| 1000/1000 [00:43<00:00, 22.96it/s]


Epoch 2 | Train Loss: 0.0103 | Val Loss: 0.3889 | LR: 0.001199


...

Epoch 5 [Val]: 100%|██████████| 13/13 [00:03<00:00,  4.20it/s, val_loss=0.1197]


Epoch 5 | Train Loss: 0.0053 | Val Loss: 0.1248 | LR: 0.001193


Sampling: 100%|██████████| 1000/1000 [00:43<00:00, 22.81it/s]


Epoch 6 | Train Loss: 0.0050 | Val Loss: 0.0974 | LR: 0.001189


Epoch 7 [Train]:  32%|███▏      | 13/41 [00:17<00:38,  1.36s/it, loss=0.0046]
